# Confluence Page Word Count
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Hash Tables, Strings, Trees · **Difficulty/Frequency:** Very Common (7/10)


## Concepts

**What this problem is really testing:**
- Tree aggregation, working bottom-up (post-order)
- Hash-map frequency counting
- Top-K selection — and the real trade-off between sorting everything vs. using a heap

**Why each one shows up here:**
- A Confluence space is a tree of pages. "Word count in a subtree" is exactly *"combine each node's own value with its children's already-combined values"* — the classic recursive tree aggregation pattern.
- Counting word frequencies is a hash-map job.
- Picking the K most frequent words is the classic top-K problem — and it has a real complexity trade-off between sorting everything vs. using a bounded heap.

**The one idea to hold onto:** never re-scan a subtree's text more than once. Compute each page's own word count exactly one time, then let the recursion combine the children's results upward — the same discipline used for "sum of a tree" or "height of a tree".

---

### Quick primers — the building blocks used below

**Tree post-order (bottom-up) aggregation.**
- In a post-order traversal, you fully process all of a node's children **before** combining their results into the node's own answer.
- Mental model: `answer(node) = combine(own_value(node), answer(child_1), answer(child_2), ...)`.
- **Cost:** each node is visited once, so total work is O(total nodes) times whatever `combine` costs per node.
- **In Python:** a plain recursive function that calls itself on `node.children`, then merges the returned values before returning its own.

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot — that's what makes insert/lookup/delete **O(1) on average** (only O(n) in rare worst-case collisions).
- **In Python:** use `dict` for key→value. Use `collections.Counter` (a `dict` subclass) specifically when the value means "how many times have I seen this key" — it comes with `.update()` for merging counts and `.most_common()` for sorted-by-frequency access.

**What is a Heap (priority queue)?**
- A heap is a tree-shaped structure (usually backed by an array) that keeps the smallest (or largest) element accessible in **O(1)**, with **O(log n)** push/pop.
- It does this by keeping one invariant true at all times: "every parent is ≤ its children" (for a min-heap).
- It never fully sorts everything — it only guarantees fast access to the *extreme* element, which is exactly what "give me the top K" needs, without paying for a full sort.
- **In Python:** the `heapq` module works on a plain `list` and is a **min-heap**. `heapq.nlargest(k, iterable, key=...)` is a ready-made "top K" helper that keeps an internal heap of size K instead of sorting everything.

**Sort-based top-K vs. heap-based top-K.**
- Sorting all U unique words costs **O(U log U)** — then slicing the top K is free.
- Keeping a heap bounded to size K costs **O(U log K)** instead — cheaper when K is much smaller than U, because the heap only ever holds K elements instead of sorting the whole vocabulary.


## Problem Statement

A Confluence space is a tree; each `Page` has `page_id`, `title`, `content` (a string), and `children` (a list of `Page`).

**Word rules:** words are alphanumeric runs (split on spaces/punctuation/newlines), case-insensitive.

- **Part 1 -- `wordCount(page) -> dict`**: frequency of each word in *that page's own content only*.
- **Part 2 -- `subtreeWordCount(page) -> dict`**: aggregated word count across the page and *all descendants*.
- **Part 3 -- `topWords(page, k, exclude) -> List[str]`**: the k most frequent words in the subtree, excluding `exclude`, ties broken alphabetically.

**Example**

```python
root = Page("p1", "Home", "the cat sat", [
    Page("p2", "Child", "the cat", []),
    Page("p3", "Child2", "sat sat", [])
])
subtreeWordCount(root)          # -> {"the": 2, "cat": 2, "sat": 3}
topWords(root, k=2, exclude=["the"])   # -> ["sat", "cat"]
```

**Official follow-up:** if pages are huge and the tree has thousands of nodes, how would you make `subtreeWordCount` more efficient with memoization or lazy evaluation?


### Approach 1 -- Single-page tokenizing (the building block)

**Idea:** `re.findall(r'[a-zA-Z0-9]+', content.lower())` extracts every alphanumeric run in one pass -- handling spaces, punctuation, and newlines uniformly, with no manual character scanning. Feed the result straight into a `Counter`.

**Time complexity:** O(M) where M is the content length -- the regex scans the string once.

**Space complexity:** O(U) where U is the number of unique words in that page.


In [ ]:
import re
from collections import Counter
from typing import List, Dict, Optional


class Page:
    def __init__(self, page_id: str, title: str, content: str, children: List["Page"]):
        self.page_id = page_id
        self.title = title
        self.content = content
        self.children = children


def wordCount(page: Page) -> Dict[str, int]:
    """Word frequency within a single page's own content."""
    words = re.findall(r"[a-zA-Z0-9]+", page.content.lower())  # one-pass tokenize + lowercase
    return dict(Counter(words))


### Approach 2 -- Naive subtree aggregation (re-tokenize concatenated text)

**Idea:** the tempting shortcut -- concatenate every descendant's content into one giant string, then tokenize *that* once. It looks like "one pass", but it silently re-walks content that Approach 1 could have already tokenized once per page, and it recomputes everything from scratch on every call (no reuse across calls from different ancestors).

**Time complexity:** O(total subtree content length) per call -- looks the same as the optimal version for a *single* call, but if `subtreeWordCount` is later called again on an ancestor or a sibling, all of the overlapping content gets re-tokenized from raw text instead of reusing an already-computed dict.

**Space complexity:** O(total subtree content length) for the concatenated string, versus O(unique words) for the optimal version -- a real, measurable waste.


In [ ]:
def _collect_content(page: Page) -> str:
    """Concatenate this page's content with every descendant's -- the naive shortcut."""
    parts = [page.content]
    for child in page.children:
        parts.append(_collect_content(child))   # re-walks every descendant's raw text
    return " ".join(parts)


def subtreeWordCount_naive(page: Page) -> Dict[str, int]:
    """Naive: tokenize the ENTIRE subtree's text in one giant re.findall call."""
    words = re.findall(r"[a-zA-Z0-9]+", _collect_content(page).lower())
    return dict(Counter(words))


### Approach 3 -- Optimal bottom-up aggregation

**Idea:** tokenize each page's *own* content exactly once (Approach 1), then combine bottom-up: `subtree(node) = own_count(node) + sum(subtree(child) for child in node.children)`. Every page's text is scanned exactly once no matter how many ancestors later ask for a subtree count that includes it (within a single top-to-bottom call).

**Time complexity:** O(N·M_avg) -- N total nodes, each page's content tokenized once; merging counters costs O(unique words per page), which is bounded by the same tokenization pass.

**Space complexity:** O(N·U_avg) for the aggregated dictionaries plus the O(depth) recursion stack.


In [ ]:
def subtreeWordCount(page: Page) -> Dict[str, int]:
    """Optimal: each page's own text is tokenized exactly once, then merged bottom-up."""
    result = Counter(wordCount(page))          # this page's own words (tokenized once)
    for child in page.children:
        result.update(subtreeWordCount(child))  # post-order: children finish before we combine
    return dict(result)


### Part 3 -- Top-K words: sort vs. heap

**Approach A (sort everything):** build `(-count, word)` tuples for every candidate and `sort()` once -- descending frequency, alphabetical tie-break, because negating the count lets a single ascending sort do both jobs.
**Time:** O(U log U). **Space:** O(U).

**Approach B (bounded heap):** when K is small relative to U, `heapq.nlargest(k, candidates)` keeps only K elements in its internal heap instead of sorting all U.
**Time:** O(U log K). **Space:** O(K).


In [ ]:
import heapq
from typing import Iterable


def topWords_sorted(page: Page, k: int, exclude: List[str]) -> List[str]:
    """Approach A: sort all candidates by (-count, word)."""
    counts = subtreeWordCount(page)
    exclude_set = set(exclude)
    candidates = [(-count, word) for word, count in counts.items() if word not in exclude_set]
    candidates.sort()                              # descending freq, then alphabetical
    return [word for _, word in candidates[:k]]


def topWords_heap(page: Page, k: int, exclude: List[str]) -> List[str]:
    """Approach B: heapq.nlargest keeps only k candidates in its internal heap."""
    counts = subtreeWordCount(page)
    exclude_set = set(exclude)
    candidates = [(-count, word) for word, count in counts.items() if word not in exclude_set]
    # nlargest on (-count, word) with the smallest tuples "largest" in our inverted scheme
    # is just nsmallest in plain terms -- easier to reuse the same tuple trick with nsmallest.
    top = heapq.nsmallest(k, candidates)
    return [word for _, word in top]


topWords = topWords_sorted   # the problem's requested `topWords` signature; sorted is the default


### Official Follow-up -- Memoization for huge trees

**Idea:** if `subtreeWordCount` is called repeatedly (e.g. once per page as a user browses, or from multiple ancestors), cache each page's result the first time it's computed. Since the recursion is a pure function of `page.page_id`'s subtree content, a cache keyed by `page_id` -- invalidated only when that page or a descendant's content changes -- turns repeated calls into O(1) lookups.

**Time complexity:** O(N·M_avg) the *first* time (same as Approach 3); O(1) for every repeated call on an already-cached subtree.

**Space complexity:** O(N·U_avg) extra for the cache -- one entry per page, same order as the uncached result.


In [ ]:
class MemoizedWordCounter:
    """Caches each page's subtree word count, keyed by page_id."""

    def __init__(self):
        self._cache: Dict[str, Dict[str, int]] = {}

    def subtreeWordCount(self, page: Page) -> Dict[str, int]:
        if page.page_id in self._cache:
            return self._cache[page.page_id]     # O(1): already computed
        result = Counter(wordCount(page))
        for child in page.children:
            result.update(self.subtreeWordCount(child))   # recursive calls also hit the cache
        result = dict(result)
        self._cache[page.page_id] = result
        return result

    def invalidate(self, page_id: str) -> None:
        """Call when a page's content changes -- caller is responsible for also invalidating ancestors."""
        self._cache.pop(page_id, None)


## Verification

Build the example tree and check every approach agrees, plus a few edge cases.

In [ ]:
root = Page("p1", "Home", "the cat sat", [
    Page("p2", "Child", "the cat", []),
    Page("p3", "Child2", "sat sat", []),
])

expected_subtree = {"the": 2, "cat": 2, "sat": 3}

assert wordCount(root) == {"the": 1, "cat": 1, "sat": 1}
assert subtreeWordCount(root) == expected_subtree
assert subtreeWordCount_naive(root) == expected_subtree   # naive is correct, just wasteful

assert topWords_sorted(root, k=2, exclude=["the"]) == ["sat", "cat"]
assert topWords_heap(root, k=2, exclude=["the"]) == ["sat", "cat"]

# Tie-break check: "cat" and "the" both appear twice -- alphabetical order must win
assert topWords_sorted(root, k=3, exclude=[]) == ["sat", "cat", "the"]

# Edge cases: empty content, leaf page, case-insensitivity, punctuation
mixed = Page("p4", "Mixed", "Hello, HELLO!! hello?", [])
assert wordCount(mixed) == {"hello": 3}
empty_leaf = Page("p5", "Empty", "", [])
assert subtreeWordCount(empty_leaf) == {}
assert topWords_sorted(empty_leaf, k=5, exclude=[]) == []

# Memoization: repeated calls reuse cached results and stay correct
counter = MemoizedWordCounter()
assert counter.subtreeWordCount(root) == expected_subtree
assert counter.subtreeWordCount(root) == expected_subtree   # second call hits the cache
counter.invalidate("p2")
assert counter.subtreeWordCount(root) == expected_subtree   # recompute p1, cache miss on p2 only

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **What if k is very large relative to unique words?** `heapq.nlargest`/`nsmallest` degrade gracefully -- for `k >= U` they're no better than sorting, so falling back to `Counter.most_common()` (which also uses a heap internally) is a reasonable simplification when K isn't known to be small.
- **Streaming / content too large for memory?** Tokenize in fixed-size chunks and feed each chunk's words into the same running `Counter` -- `Counter.update()` doesn't need to see the whole string at once, only the regex needs to not split a word across a chunk boundary (buffer a partial trailing word between chunks).
- **Very deep trees hitting Python's recursion limit?** Convert to an *iterative* post-order traversal with an explicit stack: push nodes with an "unvisited children" flag, and only combine a node's result once all of its children have been popped and combined -- shown below as a bonus.
- **Incremental updates when a page's content changes?** Invalidate the cache not just for that page but for every ancestor on the path to the root (a subtree count that includes the changed page is now stale) -- a parent-pointer walk from the changed node upward, invalidating as you go.


In [ ]:
def subtreeWordCount_iterative(root_page: Page) -> Dict[str, int]:
    """Bonus: iterative post-order aggregation -- avoids Python's recursion limit on deep trees."""
    own_counts: Dict[str, Counter] = {}
    order: List[Page] = []
    stack = [root_page]
    while stack:                       # first pass: record a valid post-order via reversed pre-order
        node = stack.pop()
        order.append(node)
        stack.extend(node.children)

    subtree: Dict[str, Dict[str, int]] = {}
    for page in reversed(order):       # children were pushed after parents, so reverse = post-order-ish
        result = Counter(wordCount(page))
        for child in page.children:
            result.update(subtree[child.page_id])
        subtree[page.page_id] = dict(result)
    return subtree[root_page.page_id]


assert subtreeWordCount_iterative(root) == expected_subtree
print("Iterative version matches the recursive one.")


## Empirical complexity check

`subtreeWordCount` (Approach 3) is O(N·M_avg): total work should scale with **total content length**, regardless of how that content is distributed across nodes. We build a wide, shallow tree (so recursion depth stays small) with N leaf pages of fixed content length, and confirm doubling N roughly doubles the time.

| Growth when n doubles | Implies |
|---|---|
| ~1x | constant / logarithmic |
| ~2x | linear |
| ~4x | quadratic |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # n leaf children under one root, each with a short, fixed-length unique-ish content
    # string -- total content length grows linearly with n, depth stays 2 (root + leaves).
    leaves = [Page(f"leaf-{i}", "Leaf", f"alpha beta gamma delta {i}", []) for i in range(n)]
    root_page = Page("root", "Root", "root content here", leaves)
    return (root_page,)

solutions = {"subtreeWordCount (optimal)": subtreeWordCount}
sizes = [2000, 4000, 8000, 16000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Post-order tree aggregation** -- compute each node's own value once, combine with children's already-computed results on the way back up. Signal: "aggregate/sum/count across a subtree" in a tree problem. Related: max depth, diameter, subtree sum, lowest common ancestor.
- **Don't re-derive data you can compute once and merge.** The naive "concatenate then tokenize" approach is correct but throws away the chance to reuse per-page work -- always ask "am I recomputing something a child already computed?" in tree aggregation.
- **`Counter` + `.update()` for merging frequency maps.** This is the idiomatic way to combine multiple frequency dicts into one, and it reads better than manual `for k, v in ... result[k] += v` loops.
- **`(-count, word)` sort key for "frequency desc, alphabetical asc".** Negating the numeric part of a sort key is the standard trick for mixing ascending and descending criteria in one `sort()`/`sorted()` call.
- **Sort vs. bounded heap for top-K.** Sort everything (O(U log U)) when you need all of it or K is close to U; use `heapq.nlargest`/`nsmallest` (O(U log K)) when K is small and fixed.
- **Memoize by a stable identity key, and invalidate along the path to the root.** A subtree aggregate is stale for every ancestor once a descendant changes, not just the changed node.
- **Recursion limit -> iterative traversal with an explicit stack.** Any post-order recursive aggregation has a mechanical iterative equivalent for arbitrarily deep trees.
- **Common pitfalls:** forgetting `.lower()` before tokenizing (breaks case-insensitivity); re-tokenizing text that's already been counted; sorting by `count` alone and getting non-deterministic tie order; caching without an invalidation story.
